# Week 4 — Applied Projects: Data Structures in Action
### Python for Blockchain Analytics | Phase 1 — Advanced Notebook

---

This notebook contains **5 complete applied projects** that combine all four
data structures together on real blockchain analytics problems.

Each project mirrors something you would actually build in production.

**Projects:**
1. Multi-chain token registry — dict + set + list
2. Wallet clustering engine — defaultdict + Counter + sets
3. DEX liquidity pool analyser — nested dicts + sorting + comprehensions
4. Block range transaction scanner — 2D lists + tuples + aggregation
5. DeFi protocol comparison dashboard — all structures combined

---

## Project 1 — Multi-Chain Token Registry

**Goal:** Build a registry that tracks the same token across multiple chains,
detects duplicates, finds cross-chain tokens, and answers lookup queries fast.

**Structures used:** `dict` (registry), `set` (chain sets), `list` (sorted results), `tuple` (token keys)

In [2]:
from collections import defaultdict

# Raw token data — as it might arrive from a multi-chain API
# Each entry = one deployment of a token on one chain
raw_token_deployments = [
    {"symbol": "USDC",  "chain": "ethereum", "address": "0xA0b8...eB48", "decimals": 6,  "is_verified": True,  "holders": 1_200_000},
    {"symbol": "USDC",  "chain": "base",     "address": "0x833...89D9",  "decimals": 6,  "is_verified": True,  "holders": 280_000},
    {"symbol": "USDC",  "chain": "arbitrum", "address": "0xFF97...3317",  "decimals": 6,  "is_verified": True,  "holders": 540_000},
    {"symbol": "ETH",   "chain": "ethereum", "address": "native",        "decimals": 18, "is_verified": True,  "holders": 5_000_000},
    {"symbol": "WETH",  "chain": "base",     "address": "0x4200...0006",  "decimals": 18, "is_verified": True,  "holders": 180_000},
    {"symbol": "WETH",  "chain": "arbitrum", "address": "0x82aF...9009",  "decimals": 18, "is_verified": True,  "holders": 320_000},
    {"symbol": "UNI",   "chain": "ethereum", "address": "0x1f98...F984",  "decimals": 18, "is_verified": True,  "holders": 310_000},
    {"symbol": "UNI",   "chain": "arbitrum", "address": "0xFa7F...8DAF",  "decimals": 18, "is_verified": True,  "holders": 45_000},
    {"symbol": "SCAM",  "chain": "ethereum", "address": "0xDead...0001",  "decimals": 18, "is_verified": False, "holders": 120},
    {"symbol": "ARB",   "chain": "arbitrum", "address": "0x912D...E4AF",  "decimals": 18, "is_verified": True,  "holders": 850_000},
    {"symbol": "AAVE",  "chain": "ethereum", "address": "0x7Fc6...C476",  "decimals": 18, "is_verified": True,  "holders": 195_000},
    {"symbol": "AAVE",  "chain": "base",     "address": "0xA700...8a33",  "decimals": 18, "is_verified": True,  "holders": 28_000},
]

# ── Step 1: Build the registry ──────────────────────────────
# Structure: { symbol → { chain → deployment_data } }
registry = defaultdict(dict)

for token in raw_token_deployments:
    symbol = token["symbol"]
    chain  = token["chain"]
    registry[symbol][chain] = {
        "address":     token["address"],
        "decimals":    token["decimals"],
        "is_verified": token["is_verified"],
        "holders":     token["holders"],
    }

# ── Step 2: Build chain sets per token ─────────────────────
# { symbol → set of chains it's on }
token_chains = {
    symbol: set(chains.keys())
    for symbol, chains in registry.items()
}

# ── Step 3: Find multi-chain tokens ─────────────────────────
multi_chain = {sym: chains for sym, chains in token_chains.items() if len(chains) > 1}
single_chain = {sym: chains for sym, chains in token_chains.items() if len(chains) == 1}

print("=" * 60)
print("  MULTI-CHAIN TOKEN REGISTRY")
print("=" * 60)

print(f"Multi-chain tokens ({len(multi_chain)}):")
for sym, chains in sorted(multi_chain.items()):
    total_holders = sum(registry[sym][c]["holders"] for c in chains)
    print(f"    {sym:<6} → {sorted(chains)} | Total holders: {total_holders:>10,}")

print(f"Single-chain tokens ({len(single_chain)}):")
for sym, chains in sorted(single_chain.items()):
    chain = list(chains)[0]
    verified = registry[sym][chain]["is_verified"]
    flag = "⚠️ unverified" if not verified else ""
    print(f"    {sym:<6} → {chain} {flag}")

# ── Step 4: Lookup queries ────────────────────────────────────
print("Lookup queries:")

# Q1: What chains is USDC on?
usdc_chains = token_chains.get("USDC", set())
print(f"    USDC chains: {sorted(usdc_chains)}")

# Q2: What tokens are on Base?
base_tokens = {sym for sym, chains in token_chains.items() if "base" in chains}
print(f"    Base tokens: {sorted(base_tokens)}")

# Q3: Tokens on both Ethereum AND Arbitrum
eth_tokens = {sym for sym, chains in token_chains.items() if "ethereum" in chains}
arb_tokens = {sym for sym, chains in token_chains.items() if "arbitrum" in chains}
on_both = eth_tokens & arb_tokens
print(f"    On Ethereum AND Arbitrum: {sorted(on_both)}")

# Q4: Get USDC address on Arbitrum
usdc_arb = registry["USDC"]["arbitrum"]["address"]
print(f"    USDC on Arbitrum: {usdc_arb}")

# ── Step 5: Sorted output by total holders ───────────────────
print("Tokens ranked by total holders:")
ranked = []
for sym, chain_data in registry.items():
    total = sum(d["holders"] for d in chain_data.values())
    chains_count = len(chain_data)
    ranked.append((sym, total, chains_count))
ranked.sort(key=lambda x: -x[1])

print(f"  {'Symbol':<8} {'Holders':>12}  {'Chains':>7}")
print("  " + "-" * 32)
for sym, total, n_chains in ranked:
    print(f"  {sym:<8} {total:>12,}  {n_chains:>7}")

  MULTI-CHAIN TOKEN REGISTRY
Multi-chain tokens (4):
    AAVE   → ['base', 'ethereum'] | Total holders:    223,000
    UNI    → ['arbitrum', 'ethereum'] | Total holders:    355,000
    USDC   → ['arbitrum', 'base', 'ethereum'] | Total holders:  2,020,000
    WETH   → ['arbitrum', 'base'] | Total holders:    500,000
Single-chain tokens (3):
    ARB    → arbitrum 
    ETH    → ethereum 
    SCAM   → ethereum ⚠️ unverified
Lookup queries:
    USDC chains: ['arbitrum', 'base', 'ethereum']
    Base tokens: ['AAVE', 'USDC', 'WETH']
    On Ethereum AND Arbitrum: ['UNI', 'USDC']
    USDC on Arbitrum: 0xFF97...3317
Tokens ranked by total holders:
  Symbol        Holders   Chains
  --------------------------------
  ETH         5,000,000        1
  USDC        2,020,000        3
  ARB           850,000        1
  WETH          500,000        2
  UNI           355,000        2
  AAVE          223,000        2
  SCAM              120        1


## Project 2 — Wallet Clustering Engine

**Goal:** Given a list of transactions, segment wallets into behaviour clusters
using only data structures.

**Structures used:** `defaultdict` (wallet profiles), `Counter` (activity counting),
`set` (unique protocols/tokens), `list` (sorted rankings)

In [3]:
from collections import defaultdict, Counter

# Simulated transaction data — what you'd get from web3.py or Etherscan
transactions = [
    {"wallet": "0xWhale1", "type": "transfer",  "value_eth": 250.0,  "protocol": None,       "token": "ETH",  "gas_gwei": 25},
    {"wallet": "0xWhale1", "type": "transfer",  "value_eth": 180.0,  "protocol": None,       "token": "ETH",  "gas_gwei": 22},
    {"wallet": "0xTrader1","type": "swap",       "value_eth": 2.5,    "protocol": "uniswap",  "token": "USDC", "gas_gwei": 30},
    {"wallet": "0xTrader1","type": "swap",       "value_eth": 1.8,    "protocol": "uniswap",  "token": "ETH",  "gas_gwei": 28},
    {"wallet": "0xTrader1","type": "swap",       "value_eth": 3.2,    "protocol": "curve",    "token": "USDC", "gas_gwei": 32},
    {"wallet": "0xTrader1","type": "swap",       "value_eth": 0.9,    "protocol": "uniswap",  "token": "UNI",  "gas_gwei": 27},
    {"wallet": "0xLP1",    "type": "liquidity",  "value_eth": 15.0,   "protocol": "uniswap",  "token": "ETH",  "gas_gwei": 45},
    {"wallet": "0xLP1",    "type": "liquidity",  "value_eth": 12.5,   "protocol": "curve",    "token": "USDC", "gas_gwei": 40},
    {"wallet": "0xLP1",    "type": "swap",       "value_eth": 1.2,    "protocol": "uniswap",  "token": "ETH",  "gas_gwei": 28},
    {"wallet": "0xBot1",   "type": "swap",       "value_eth": 0.5,    "protocol": "uniswap",  "token": "ETH",  "gas_gwei": 150},
    {"wallet": "0xBot1",   "type": "swap",       "value_eth": 0.5,    "protocol": "uniswap",  "token": "ETH",  "gas_gwei": 148},
    {"wallet": "0xBot1",   "type": "swap",       "value_eth": 0.5,    "protocol": "uniswap",  "token": "ETH",  "gas_gwei": 152},
    {"wallet": "0xBot1",   "type": "swap",       "value_eth": 0.5,    "protocol": "uniswap",  "token": "ETH",  "gas_gwei": 149},
    {"wallet": "0xBot1",   "type": "swap",       "value_eth": 0.5,    "protocol": "uniswap",  "token": "ETH",  "gas_gwei": 151},
    {"wallet": "0xRetail1","type": "transfer",   "value_eth": 0.05,   "protocol": None,       "token": "ETH",  "gas_gwei": 18},
    {"wallet": "0xRetail1","type": "swap",       "value_eth": 0.12,   "protocol": "uniswap",  "token": "USDC", "gas_gwei": 20},
    {"wallet": "0xNFT1",   "type": "NFT_buy",    "value_eth": 1.5,    "protocol": "opensea",  "token": "ETH",  "gas_gwei": 35},
    {"wallet": "0xNFT1",   "type": "NFT_buy",    "value_eth": 2.8,    "protocol": "blur",     "token": "ETH",  "gas_gwei": 38},
    {"wallet": "0xNFT1",   "type": "NFT_sell",   "value_eth": 3.2,    "protocol": "blur",     "token": "ETH",  "gas_gwei": 33},
]

# ── Step 1: Build wallet profiles ────────────────────────────
profiles = defaultdict(lambda: {
    "tx_count":        0,
    "total_volume":    0.0,
    "max_tx":          0.0,
    "gas_prices":      [],
    "type_counter":    Counter(),
    "protocols":       set(),
    "tokens":          set(),
})

for tx in transactions:
    w = tx["wallet"]
    p = profiles[w]

    p["tx_count"]     += 1
    p["total_volume"] += tx["value_eth"]
    p["max_tx"]        = max(p["max_tx"], tx["value_eth"])
    p["gas_prices"].append(tx["gas_gwei"])
    p["type_counter"][tx["type"]] += 1
    p["tokens"].add(tx["token"])
    if tx["protocol"]:
        p["protocols"].add(tx["protocol"])

# ── Step 2: Derive features ──────────────────────────────────
def derive_features(wallet, profile):
    gas     = profile["gas_prices"]
    avg_gas = sum(gas) / len(gas)
    gas_std = (sum((g - avg_gas)**2 for g in gas) / len(gas)) ** 0.5
    return {
        "wallet":         wallet,
        "tx_count":       profile["tx_count"],
        "total_volume":   profile["total_volume"],
        "max_tx":         profile["max_tx"],
        "avg_gas":        avg_gas,
        "gas_std":        gas_std,          # low std = bot-like uniformity
        "unique_protocols": len(profile["protocols"]),
        "unique_tokens":  len(profile["tokens"]),
        "swap_ratio":     profile["type_counter"]["swap"] / profile["tx_count"],
        "lp_ratio":       profile["type_counter"]["liquidity"] / profile["tx_count"],
        "nft_ratio":      (profile["type_counter"]["NFT_buy"] + profile["type_counter"]["NFT_sell"]) / profile["tx_count"],
        "top_type":       profile["type_counter"].most_common(1)[0][0],
    }

features = [derive_features(w, p) for w, p in profiles.items()]

# ── Step 3: Rule-based clustering ────────────────────────────
def classify_wallet(f):
    # Bot: high gas, low gas variance, repetitive same-size swaps
    if f["avg_gas"] > 100 and f["gas_std"] < 5:
        return "🤖 MEV Bot"
    # Whale: any single tx > 100 ETH
    if f["max_tx"] >= 100:
        return "🐋 Whale"
    # LP: majority of activity is liquidity provision
    if f["lp_ratio"] >= 0.3:
        return "💧 Liquidity Provider"
    # NFT: majority of activity is NFT trading
    if f["nft_ratio"] >= 0.5:
        return "🎨 NFT Trader"
    # DeFi power user: many swaps, multiple protocols
    if f["swap_ratio"] >= 0.5 and f["unique_protocols"] >= 2:
        return "⚡ DeFi Power User"
    # Retail: low volume, few txns
    if f["total_volume"] < 1.0 and f["tx_count"] <= 3:
        return "👤 Retail User"
    return "🔍 Unclassified"

# ── Step 4: Print cluster report ─────────────────────────────
print("=" * 65)
print("  WALLET CLUSTER REPORT")
print("=" * 65)
print(f"  {'Wallet':<12} {'Cluster':<22} {'Txns':>5}  {'Volume':>10}  {'Avg Gas':>8}")
print("  " + "-" * 63)

for f in sorted(features, key=lambda x: -x["total_volume"]):
    cluster = classify_wallet(f)
    print(f"  {f['wallet']:<12} {cluster:<22} {f['tx_count']:>5}  "
          f"{f['total_volume']:>9.2f}E  {f['avg_gas']:>7.0f}G")

# ── Step 5: Cluster summary using Counter ────────────────────
cluster_counts = Counter(classify_wallet(f) for f in features)
print("Cluster distribution:")
for cluster, count in cluster_counts.most_common():
    print(f"    {cluster:<22} {count} wallet(s)")

  WALLET CLUSTER REPORT
  Wallet       Cluster                 Txns      Volume   Avg Gas
  ---------------------------------------------------------------
  0xWhale1     🐋 Whale                    2     430.00E       24G
  0xLP1        💧 Liquidity Provider       3      28.70E       38G
  0xTrader1    ⚡ DeFi Power User          4       8.40E       29G
  0xNFT1       🎨 NFT Trader               3       7.50E       35G
  0xBot1       🤖 MEV Bot                  5       2.50E      150G
  0xRetail1    👤 Retail User              2       0.17E       19G
Cluster distribution:
    🐋 Whale                1 wallet(s)
    ⚡ DeFi Power User      1 wallet(s)
    💧 Liquidity Provider   1 wallet(s)
    🤖 MEV Bot              1 wallet(s)
    👤 Retail User          1 wallet(s)
    🎨 NFT Trader           1 wallet(s)


## Project 3 — DEX Liquidity Pool Analyser

**Goal:** Rank, filter and score Uniswap v3 pools from a registry of pool data.

**Structures used:** `list` of `dict` (pool data), `sorted()` with complex keys,
`dict` comprehensions, nested access

In [ ]:
# Uniswap v3 pool data (what TheGraph or a custom indexer returns)
pools = [
    {"id":"0x88e6","token0":"USDC","token1":"ETH", "fee":500,  "tvl":800_000_000,"vol24h":420_000_000,"vol7d":2_800_000_000,"tx24h":18_420},
    {"id":"0x8ad5","token0":"USDC","token1":"ETH", "fee":3000, "tvl":450_000_000,"vol24h":180_000_000,"vol7d":1_100_000_000,"tx24h":9_210},
    {"id":"0x4e68","token0":"ETH", "token1":"USDT","fee":3000, "tvl":320_000_000,"vol24h":95_000_000, "vol7d":620_000_000, "tx24h":4_800},
    {"id":"0xcbcf","token0":"WBTC","token1":"ETH", "fee":3000, "tvl":280_000_000,"vol24h":65_000_000, "vol7d":430_000_000, "tx24h":2_100},
    {"id":"0x6c6b","token0":"USDC","token1":"USDT","fee":100,  "tvl":210_000_000,"vol24h":380_000_000,"vol7d":2_400_000_000,"tx24h":22_000},
    {"id":"0x7858","token0":"DAI", "token1":"USDC","fee":100,  "tvl":180_000_000,"vol24h":290_000_000,"vol7d":1_900_000_000,"tx24h":15_300},
    {"id":"0x1d42","token0":"UNI", "token1":"ETH", "fee":3000, "tvl":42_000_000, "vol24h":12_000_000, "vol7d":78_000_000,  "tx24h":1_250},
    {"id":"0x3416","token0":"AAVE","token1":"ETH", "fee":3000, "tvl":28_000_000, "vol24h":8_000_000,  "vol7d":51_000_000,  "tx24h":820},
    {"id":"0x290a","token0":"ARB", "token1":"ETH", "fee":3000, "tvl":85_000_000, "vol24h":31_000_000, "vol7d":210_000_000, "tx24h":3_400},
    {"id":"0x11b8","token0":"CRV", "token1":"ETH", "fee":3000, "tvl":15_000_000, "vol24h":4_200_000,  "vol7d":29_000_000,  "tx24h":580},
]

STABLECOINS = {"USDC", "USDT", "DAI", "FRAX", "LUSD"}

# ── Step 1: Enrich each pool with derived metrics ─────────────
def enrich_pool(pool):
    fee_pct     = pool["fee"] / 1_000_000          # fee in decimal (500 → 0.0005)
    fees_24h    = pool["vol24h"] * fee_pct
    fees_7d     = pool["vol7d"]  * fee_pct
    apr_est     = (fees_7d * 52) / pool["tvl"] * 100   # annualised from 7d fees
    vol_tvl     = pool["vol24h"] / pool["tvl"]          # capital efficiency
    is_stable   = pool["token0"] in STABLECOINS and pool["token1"] in STABLECOINS
    pair        = f"{pool['token0']}/{pool['token1']}"

    return {**pool,
        "pair":      pair,
        "fee_pct":   fee_pct,
        "fees_24h":  fees_24h,
        "apr_est":   apr_est,
        "vol_tvl":   vol_tvl,
        "is_stable": is_stable,
    }

enriched = [enrich_pool(p) for p in pools]

# ── Step 2: Rankings ─────────────────────────────────────────
by_tvl      = sorted(enriched, key=lambda p: -p["tvl"])
by_volume   = sorted(enriched, key=lambda p: -p["vol24h"])
by_apr      = sorted(enriched, key=lambda p: -p["apr_est"])
by_efficiency = sorted(enriched, key=lambda p: -p["vol_tvl"])

def print_ranking(title, ranked, key, fmt):
    print(f"
  {title}")
    print(f"  {'Rank':<5} {'Pair':<14} {'Fee':>6}  {title.split()[1]:>14}")
    print("  " + "-" * 45)
    for i, pool in enumerate(ranked[:5], 1):
        val = fmt(pool[key])
        print(f"  {i:<5} {pool['pair']:<14} {pool['fee']/10000:.2f}%  {val:>14}")

print("=" * 50)
print("  DEX POOL ANALYSER — Uniswap V3")
print("=" * 50)

print_ranking("TVL Rankings",      by_tvl,      "tvl",     lambda v: f"${v/1e6:,.0f}M")
print_ranking("Volume Rankings",   by_volume,   "vol24h",  lambda v: f"${v/1e6:,.0f}M")
print_ranking("APR Rankings",      by_apr,      "apr_est", lambda v: f"{v:.1f}%")
print_ranking("Efficiency Ranking",by_efficiency,"vol_tvl", lambda v: f"{v:.2f}x")

# ── Step 3: Filters ──────────────────────────────────────────
print("
  Stable-stable pools (lowest IL risk):")
stable_pools = [p for p in enriched if p["is_stable"]]
for p in sorted(stable_pools, key=lambda x: -x["vol24h"]):
    print(f"    {p['pair']:<14} TVL: ${p['tvl']/1e6:.0f}M  APR: {p['apr_est']:.1f}%")

print("
  High-yield volatile pools (>15% APR, TVL>$50M):")
high_yield = [p for p in enriched if p["apr_est"] > 15 and p["tvl"] > 50_000_000 and not p["is_stable"]]
for p in sorted(high_yield, key=lambda x: -x["apr_est"]):
    print(f"    {p['pair']:<14} TVL: ${p['tvl']/1e6:.0f}M  APR: {p['apr_est']:.1f}%  Vol: ${p['vol24h']/1e6:.0f}M/d")

# ── Step 4: Fee tier breakdown using defaultdict ──────────────
from collections import defaultdict
by_fee_tier = defaultdict(list)
for p in enriched:
    by_fee_tier[p["fee"]].append(p)

print("
  Pools by fee tier:")
for fee in sorted(by_fee_tier.keys()):
    tier_pools = by_fee_tier[fee]
    total_tvl  = sum(p["tvl"]    for p in tier_pools)
    total_vol  = sum(p["vol24h"] for p in tier_pools)
    print(f"    {fee/10000:.2f}% tier: {len(tier_pools)} pools  |  "
          f"TVL: ${total_tvl/1e9:.2f}B  |  Vol: ${total_vol/1e6:.0f}M/d")

## Project 4 — Block Range Transaction Scanner

**Goal:** Scan a simulated range of blocks, aggregate stats, and flag anomalies.

**Structures used:** `list` of `tuples` (block data), 2D aggregation, `set` (unique senders), `Counter`

In [ ]:
from collections import Counter
import random

random.seed(42)

# Simulate block data — what web3.eth.get_block(n, full_transactions=True) returns
# Each block has a list of transactions
WALLETS = ["0xWhale1","0xTrader1","0xBot1","0xBot2","0xRetail1",
           "0xRetail2","0xLP1","0xArb1","0xNFT1","0xBridge1"]

def simulate_block(block_number):
    n_txns   = random.randint(80, 300)
    base_fee = random.uniform(10, 80)  # gwei
    txns     = []
    for _ in range(n_txns):
        sender     = random.choice(WALLETS)
        value_eth  = random.choice([0, 0, 0, 0.1, 0.5, 1.0, 5.0, 50.0, 200.0])
        gas_price  = base_fee + random.uniform(-5, 20)
        tx_type    = random.choice(["transfer","swap","swap","swap","liquidity","NFT","contract"])
        txns.append((sender, round(value_eth, 4), round(gas_price, 1), tx_type))
    return {
        "number":    block_number,
        "base_fee":  round(base_fee, 1),
        "tx_count":  n_txns,
        "txns":      txns,              # list of (sender, value_eth, gas_gwei, type)
    }

# Scan 20 blocks
START_BLOCK = 19_847_000
N_BLOCKS    = 20
blocks      = [simulate_block(START_BLOCK + i) for i in range(N_BLOCKS)]

# ── Aggregate across all blocks ──────────────────────────────
total_txns       = 0
total_volume_eth = 0.0
unique_senders   = set()
type_counter     = Counter()
block_summaries  = []
gas_prices_all   = []
whale_txns       = []     # value_eth > 50

for block in blocks:
    block_volume = 0.0
    block_fees   = []

    for sender, value_eth, gas_gwei, tx_type in block["txns"]:
        total_txns       += 1
        total_volume_eth += value_eth
        block_volume     += value_eth
        unique_senders.add(sender)
        type_counter[tx_type] += 1
        gas_prices_all.append(gas_gwei)
        block_fees.append(gas_gwei)

        if value_eth >= 50:
            whale_txns.append({
                "block":     block["number"],
                "sender":    sender,
                "value_eth": value_eth,
                "type":      tx_type,
            })

    block_summaries.append({
        "block":       block["number"],
        "tx_count":    block["tx_count"],
        "volume_eth":  round(block_volume, 4),
        "base_fee":    block["base_fee"],
        "avg_gas":     round(sum(block_fees) / len(block_fees), 1),
    })

# ── Compute stats ─────────────────────────────────────────────
avg_gas     = sum(gas_prices_all) / len(gas_prices_all)
max_gas     = max(gas_prices_all)
min_gas     = min(gas_prices_all)
avg_block_vol = total_volume_eth / N_BLOCKS

# ── Print summary ─────────────────────────────────────────────
print("=" * 62)
print(f"  BLOCK SCAN REPORT  |  Blocks {START_BLOCK:,} – {START_BLOCK+N_BLOCKS-1:,}")
print("=" * 62)
print(f"  Blocks scanned   : {N_BLOCKS}")
print(f"  Total txns       : {total_txns:,}")
print(f"  Unique senders   : {len(unique_senders)}")
print(f"  Total volume     : {total_volume_eth:,.2f} ETH")
print(f"  Avg block volume : {avg_block_vol:,.2f} ETH")
print(f"  Gas range        : {min_gas:.1f} – {max_gas:.1f} Gwei (avg: {avg_gas:.1f})")

print(f"
  Transaction types:")
for tx_type, count in type_counter.most_common():
    pct = count / total_txns * 100
    bar = "█" * int(pct / 2)
    print(f"    {tx_type:<12} {count:>5,} ({pct:>4.1f}%)  {bar}")

print(f"
  Block-by-block summary:")
print(f"  {'Block':>12}  {'Txns':>5}  {'Volume (ETH)':>14}  {'Base Fee':>9}  {'Avg Gas':>8}")
print("  " + "-" * 55)
for b in block_summaries:
    vol_flag = " 🚨" if b["volume_eth"] > avg_block_vol * 2 else ""
    print(f"  {b['block']:>12,}  {b['tx_count']:>5}  {b['volume_eth']:>14,.2f}  "
          f"{b['base_fee']:>8.1f}G  {b['avg_gas']:>7.1f}G{vol_flag}")

if whale_txns:
    print(f"
  🐋 Whale transactions (≥50 ETH):")
    for tx in sorted(whale_txns, key=lambda x: -x["value_eth"]):
        print(f"    Block {tx['block']:,} | {tx['sender']} | {tx['value_eth']:.1f} ETH | {tx['type']}")

## Project 5 — DeFi Protocol Comparison Dashboard

**Goal:** Compare multiple DeFi protocols across TVL, volume, fees, and user activity.
Produce a ranked comparison table and identify leaders in each category.

**Structures used:** All four — `list` of `dict`, nested `dict`, `set` operations,
`Counter`, complex `sorted()`, dict comprehensions

In [ ]:
from collections import Counter, defaultdict

protocols = [
    {
        "name": "Uniswap V3",
        "type": "DEX",
        "chains": ["ethereum","arbitrum","optimism","base","polygon"],
        "tvl_usd":     5_200_000_000,
        "volume_24h":  1_800_000_000,
        "fees_24h":    5_400_000,
        "users_24h":   48_000,
        "tx_24h":      92_000,
        "token":       "UNI",
        "token_price": 12.84,
        "audits":      ["Trail of Bits","OpenZeppelin","ABDK"],
        "launched":    2021,
    },
    {
        "name": "Aave V3",
        "type": "Lending",
        "chains": ["ethereum","arbitrum","optimism","base","polygon","avalanche"],
        "tvl_usd":     12_800_000_000,
        "volume_24h":  480_000_000,
        "fees_24h":    1_440_000,
        "users_24h":   8_200,
        "tx_24h":      14_000,
        "token":       "AAVE",
        "token_price": 98.50,
        "audits":      ["OpenZeppelin","SigmaPrime","Peckshield"],
        "launched":    2020,
    },
    {
        "name": "Curve Finance",
        "type": "DEX",
        "chains": ["ethereum","arbitrum","optimism","polygon","avalanche","fantom"],
        "tvl_usd":     2_100_000_000,
        "volume_24h":  420_000_000,
        "fees_24h":    420_000,
        "users_24h":   5_800,
        "tx_24h":      9_200,
        "token":       "CRV",
        "token_price": 0.48,
        "audits":      ["Trail of Bits","Quantstamp"],
        "launched":    2020,
    },
    {
        "name": "Lido",
        "type": "Liquid Staking",
        "chains": ["ethereum","polygon","solana"],
        "tvl_usd":     32_000_000_000,
        "volume_24h":  120_000_000,
        "fees_24h":    960_000,
        "users_24h":   3_100,
        "tx_24h":      5_400,
        "token":       "LDO",
        "token_price": 2.14,
        "audits":      ["Sigma Prime","MixBytes","Quantstamp","OpenZeppelin"],
        "launched":    2020,
    },
    {
        "name": "GMX",
        "type": "Perps DEX",
        "chains": ["arbitrum","avalanche"],
        "tvl_usd":     680_000_000,
        "volume_24h":  310_000_000,
        "fees_24h":    930_000,
        "users_24h":   4_200,
        "tx_24h":      7_800,
        "token":       "GMX",
        "token_price": 38.50,
        "audits":      ["ABDK","Quantstamp"],
        "launched":    2021,
    },
    {
        "name": "MakerDAO",
        "type": "CDP / Stablecoin",
        "chains": ["ethereum"],
        "tvl_usd":     8_400_000_000,
        "volume_24h":  95_000_000,
        "fees_24h":    285_000,
        "users_24h":   1_800,
        "tx_24h":      3_200,
        "token":       "MKR",
        "token_price": 2850.0,
        "audits":      ["Trail of Bits","Runtime Verification","Peckshield"],
        "launched":    2017,
    },
]

# ── Enrich with derived metrics ───────────────────────────────
for p in protocols:
    p["fee_yield_pct"]   = p["fees_24h"] / p["tvl_usd"] * 365 * 100   # annualised fee yield
    p["vol_tvl_ratio"]   = p["volume_24h"] / p["tvl_usd"]
    p["fee_per_user"]    = p["fees_24h"] / p["users_24h"] if p["users_24h"] else 0
    p["chain_count"]     = len(p["chains"])
    p["audit_count"]     = len(p["audits"])
    p["revenue_per_tx"]  = p["fees_24h"] / p["tx_24h"] if p["tx_24h"] else 0
    p["years_live"]      = 2026 - p["launched"]

# ── Category leaders ─────────────────────────────────────────
def leader(metric, reverse=True):
    ranked = sorted(protocols, key=lambda p: p[metric], reverse=reverse)
    return ranked[0]["name"], ranked[0][metric]

categories = [
    ("tvl_usd",        "Highest TVL",          lambda v: f"${v/1e9:.1f}B"),
    ("volume_24h",     "Highest 24h Volume",   lambda v: f"${v/1e6:.0f}M"),
    ("fees_24h",       "Highest 24h Fees",     lambda v: f"${v/1e6:.2f}M"),
    ("fee_yield_pct",  "Best Fee APY",         lambda v: f"{v:.2f}%"),
    ("users_24h",      "Most Active Users",    lambda v: f"{v:,}"),
    ("chain_count",    "Most Multi-chain",     lambda v: f"{v} chains"),
    ("audit_count",    "Most Audited",         lambda v: f"{v} audits"),
]

print("=" * 62)
print("  DEFI PROTOCOL COMPARISON DASHBOARD")
print("=" * 62)

print("
  🏆 Category Leaders:")
for metric, label, fmt in categories:
    name, val = leader(metric)
    print(f"    {label:<25} → {name:<15}  {fmt(val)}")

# ── Full comparison table ─────────────────────────────────────
by_tvl = sorted(protocols, key=lambda p: -p["tvl_usd"])
print(f"
  {'Protocol':<15} {'Type':<16} {'TVL':>9}  {'Vol/24h':>8}  {'Fee APY':>8}  {'Users':>7}  {'Chains':>7}")
print("  " + "-" * 76)
for p in by_tvl:
    print(f"  {p['name']:<15} {p['type']:<16} "
          f"${p['tvl_usd']/1e9:>7.1f}B  "
          f"${p['volume_24h']/1e6:>6.0f}M  "
          f"{p['fee_yield_pct']:>7.2f}%  "
          f"{p['users_24h']:>7,}  "
          f"{p['chain_count']:>7}")

# ── Chain coverage analysis ───────────────────────────────────
all_chains_by_protocol = {p["name"]: set(p["chains"]) for p in protocols}
all_chains = set()
for chains in all_chains_by_protocol.values():
    all_chains |= chains

print(f"
  Chain coverage ({len(all_chains)} unique chains):")
for chain in sorted(all_chains):
    on_chain = [p["name"] for p in protocols if chain in p["chains"]]
    print(f"    {chain:<12} ({len(on_chain)} protocols): {', '.join(on_chain)}")

# ── Protocol type breakdown ───────────────────────────────────
type_counter = Counter(p["type"] for p in protocols)
type_tvl     = defaultdict(float)
for p in protocols:
    type_tvl[p["type"]] += p["tvl_usd"]

print(f"
  TVL by protocol type:")
for ptype, tvl in sorted(type_tvl.items(), key=lambda x: -x[1]):
    count = type_counter[ptype]
    print(f"    {ptype:<20} ${tvl/1e9:.1f}B  ({count} protocol{'s' if count > 1 else ''})")

# ── Audit firms ───────────────────────────────────────────────
audit_counter = Counter()
for p in protocols:
    for auditor in p["audits"]:
        audit_counter[auditor] += 1

print(f"
  Most trusted audit firms (by protocol coverage):")
for firm, count in audit_counter.most_common():
    audited = [p["name"] for p in protocols if firm in p["audits"]]
    print(f"    {firm:<22} {count} protocols: {', '.join(audited)}")

---

## What you've built in this notebook

Five production-grade analytics tools using only Python data structures:

| Project | What it does | Key structures |
|---------|-------------|----------------|
| Token Registry | Multi-chain token lookup + holder stats | `defaultdict`, `set`, `dict` |
| Wallet Clustering | Behaviour segmentation from tx data | `defaultdict`, `Counter`, `set` |
| Pool Analyser | DEX pool ranking + APR calculation | `list` of `dict`, `sorted(key=)` |
| Block Scanner | Multi-block aggregation + anomaly detection | `tuple`, `set`, `Counter`, 2D |
| Protocol Dashboard | Cross-protocol comparison + chain coverage | All four structures combined |

These patterns directly transfer to:
- Phase 3 (web3.py + on-chain data)
- Phase 5 (ETL pipelines — the flatten pattern is exactly what dbt does)
- Phase 7 (ML — feature dictionaries feed directly into scikit-learn)

---

Now work through `exercises_advanced.py` — 8 hard problems that test everything.